In [3]:
!pip install torch --index-url https://download.pytorch.org/whl/cu121
!pip install git+https://github.com/k2-fsa/OmniVoice.git
!pip install peft accelerate soundfile

Looking in indexes: https://download.pytorch.org/whl/cu121
  Cloning https://github.com/k2-fsa/OmniVoice.git to /tmp/pip-req-build-ittojn0u
  Running command git clone --filter=blob:none --quiet https://github.com/k2-fsa/OmniVoice.git /tmp/pip-req-build-ittojn0u
  Resolved https://github.com/k2-fsa/OmniVoice.git to commit 38e992bc60f85548faeb77e8fa70158ba71deb30
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# ---------- الخلية 2: تحميل الموديل الأساسي ----------


In [4]:
from voicetut_tts import VoiceTutTTS

tts = VoiceTutTTS.from_pretrained("mohammedaly22/VoiceTut-TTS")
print("الموديل اتحمل بنجاح")

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

الموديل اتحمل بنجاح


In [40]:
import subprocess, sys, os

manifest = "/kaggle/working/md1_train_fixed.jsonl"   # اللي كنت شغّال عليه
token_dir = "/kaggle/working/md1_tokens"
train_out = "/kaggle/working/finetuned_model"

# 1. استخراج الـ tokens (الخطوة اللي كانت بتفشل بسبب k2-fsa/OmniVoice)
cmd_extract = [
    sys.executable, "-m", "omnivoice.scripts.extract_audio_tokens",
    "--input_jsonl", manifest,
    "--tar_output_pattern", f"{token_dir}/train/audios/shard-%06d.tar",
    "--jsonl_output_pattern", f"{token_dir}/train/txts/shard-%06d.jsonl",
    "--tokenizer_path", "mohammedaly22/VoiceTut-TTS",   # هذا النموذج يحتوي الـ feature extractor
    "--nj_per_gpu", "1",
    "--shuffle", "False"
]
print("⏳ جاري استخراج الـ tokens (لن يأخذ وقتاً طويلاً)...")
subprocess.run(cmd_extract, cwd="/kaggle/working/OmniVoice", check=True)
print("✅ تم استخراج الـ tokens بنجاح")

# 2. تشغيل التدريب (LoRA fine‑tuning)  
# نبحث عن أول ملف jsonl خرج من الاستخراج
import glob
txt_files = sorted(glob.glob(f"{token_dir}/train/txts/shard-*.jsonl"))
if not txt_files:
    raise FileNotFoundError("مافيش ملفات txts طلعت من الاستخراج")
train_manifest = txt_files[0]

cmd_train = [
    sys.executable, "-m", "omnivoice.scripts.train",
    "--model_name_or_path", "mohammedaly22/VoiceTut-TTS",
    "--tokenizer_path", "mohammedaly22/VoiceTut-TTS",
    "--train_manifest", train_manifest,
    "--output_dir", train_out,
    "--use_lora", "True",
    "--batch_size", "1",
    "--gradient_accumulation_steps", "4",
    "--learning_rate", "2e-5",
    "--num_epochs", "40",
    "--save_steps", "20",
    "--logging_steps", "5",
    "--fp16", "True"
]
print("🚀 بدء التدريب...")
subprocess.run(cmd_train, cwd="/kaggle/working/OmniVoice", check=True)
print(f"🎉 التدريب خلص! النموذج الجاهز محفوظ في: {train_out}")

⏳ جاري استخراج الـ tokens (لن يأخذ وقتاً طويلاً)...


2026-08-11 18:39:38,633 INFO [extract_audio_tokens.py:341] Input mode: raw JSONL (/kaggle/working/md1_train_fixed.jsonl)
2026-08-11 18:39:38,634 INFO [extract_audio_tokens.py:399] Adjusted samples_per_shard from 1000 to 1 to meet min_num_shards=32 (total_samples=1)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 24 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
2026-08-11 18:39:38,667 INFO [extract_audio_tokens.py:427] GPU count: 2, Processes per GPU: 1, Total processes: 2
Extracting Audio Tokens:   0%|          | 0/1 [00:00<?, ?it/s]2026-08-11 18:39:46,436 INFO [extract_audio_tokens.py:548] Submitting tasks...

CalledProcessError: Command '['/usr/bin/python3', '-m', 'omnivoice.scripts.extract_audio_tokens', '--input_jsonl', '/kaggle/working/md1_train_fixed.jsonl', '--tar_output_pattern', '/kaggle/working/md1_tokens/train/audios/shard-%06d.tar', '--jsonl_output_pattern', '/kaggle/working/md1_tokens/train/txts/shard-%06d.jsonl', '--tokenizer_path', 'mohammedaly22/VoiceTut-TTS', '--nj_per_gpu', '1', '--shuffle', 'False']' returned non-zero exit status 1.

In [42]:
import os, subprocess, sys, shutil
from huggingface_hub import snapshot_download

# 1. تحميل مجلد audio_tokenizer من k2-fsa/OmniVoice إلى مجلد محلي
local_tokenizer_path = "/kaggle/working/audio_tokenizer_local"
if not os.path.exists(local_tokenizer_path):
    print("⏳ جاري تحميل الـ audio_tokenizer (مرة واحدة فقط)...")
    snapshot_download(
        repo_id="k2-fsa/OmniVoice",
        allow_patterns="audio_tokenizer/*",
        local_dir=local_tokenizer_path,
        local_dir_use_symlinks=False
    )
    # الملفات هتكون جوه audio_tokenizer/ بداخل المسار المحلي، ننقلهم للأعلى عشان المسار المباشر
    inner_dir = os.path.join(local_tokenizer_path, "audio_tokenizer")
    for f in os.listdir(inner_dir):
        shutil.move(os.path.join(inner_dir, f), os.path.join(local_tokenizer_path, f))
    shutil.rmtree(inner_dir)
    print("✅ تم تجهيز الـ tokenizer محليًا")

# 2. استخراج الـ tokens باستخدام المسار المحلي
manifest = "/kaggle/working/md1_train_fixed.jsonl"
token_dir = "/kaggle/working/md1_tokens"
os.makedirs(token_dir, exist_ok=True)

cmd_extract = [
    sys.executable, "-m", "omnivoice.scripts.extract_audio_tokens",
    "--input_jsonl", manifest,
    "--tar_output_pattern", f"{token_dir}/train/audios/shard-%06d.tar",
    "--jsonl_output_pattern", f"{token_dir}/train/txts/shard-%06d.jsonl",
    "--tokenizer_path", local_tokenizer_path,   # المسار المحلي اللي فيه preprocessor_config.json
    "--nj_per_gpu", "1",
    "--shuffle", "False"
]
print("🚀 بدء استخراج الـ tokens...")
subprocess.run(cmd_extract, cwd="/kaggle/working/OmniVoice", check=True)
print("✅ تم استخراج الـ tokens بنجاح")

# 3. بدء التدريب (باستخدام الموديل Mohammedaly22/VoiceTut-TTS)
import glob
txt_files = sorted(glob.glob(f"{token_dir}/train/txts/shard-*.jsonl"))
if not txt_files:
    raise FileNotFoundError("لا توجد ملفات txts بعد الاستخراج")
train_manifest = txt_files[0]

cmd_train = [
    sys.executable, "-m", "omnivoice.scripts.train",
    "--model_name_or_path", "mohammedaly22/VoiceTut-TTS",
    "--tokenizer_path", local_tokenizer_path,  # نفس الـ tokenizer المحلي لتدريب LoRA
    "--train_manifest", train_manifest,
    "--output_dir", "/kaggle/working/finetuned_model",
    "--use_lora", "True",
    "--batch_size", "1",
    "--gradient_accumulation_steps", "4",
    "--learning_rate", "2e-5",
    "--num_epochs", "40",
    "--save_steps", "20",
    "--logging_steps", "5",
    "--fp16", "True"
]
print("🏋️ بدء التدريب...")
subprocess.run(cmd_train, cwd="/kaggle/working/OmniVoice", check=True)
print("🎉 التدريب خلص! النموذج النهائي في /kaggle/working/finetuned_model")

⏳ جاري تحميل الـ audio_tokenizer (مرة واحدة فقط)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

✅ تم تجهيز الـ tokenizer محليًا
🚀 بدء استخراج الـ tokens...


2026-08-11 18:47:38,518 INFO [extract_audio_tokens.py:341] Input mode: raw JSONL (/kaggle/working/md1_train_fixed.jsonl)
2026-08-11 18:47:38,518 INFO [extract_audio_tokens.py:399] Adjusted samples_per_shard from 1000 to 1 to meet min_num_shards=32 (total_samples=1)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 24 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
2026-08-11 18:47:38,550 INFO [extract_audio_tokens.py:427] GPU count: 2, Processes per GPU: 1, Total processes: 2
Extracting Audio Tokens:   0%|          | 0/1 [00:00<?, ?it/s]2026-08-11 18:47:46,175 INFO [extract_audio_tokens.py:548] Submitting tasks...

✅ تم استخراج الـ tokens بنجاح
🏋️ بدء التدريب...


/usr/bin/python3: No module named omnivoice.scripts.train


CalledProcessError: Command '['/usr/bin/python3', '-m', 'omnivoice.scripts.train', '--model_name_or_path', 'mohammedaly22/VoiceTut-TTS', '--tokenizer_path', '/kaggle/working/audio_tokenizer_local', '--train_manifest', '/kaggle/working/md1_tokens/train/txts/shard-000000.jsonl', '--output_dir', '/kaggle/working/finetuned_model', '--use_lora', 'True', '--batch_size', '1', '--gradient_accumulation_steps', '4', '--learning_rate', '2e-5', '--num_epochs', '40', '--save_steps', '20', '--logging_steps', '5', '--fp16', 'True']' returned non-zero exit status 1.

In [45]:
import os
for root, dirs, files in os.walk("/kaggle/working/OmniVoice/omnivoice"):
    for f in files:
        if "train" in f.lower() and f.endswith(".py"):
            print(os.path.join(root, f))

/kaggle/working/OmniVoice/omnivoice/cli/train.py
/kaggle/working/OmniVoice/omnivoice/training/trainer.py


In [47]:
import subprocess, sys, glob, os

token_dir = "/kaggle/working/md1_tokens"
train_manifest = sorted(glob.glob(f"{token_dir}/train/txts/shard-*.jsonl"))[0]
local_tokenizer_path = "/kaggle/working/audio_tokenizer_local"
output_dir = "/kaggle/working/finetuned_model"

cmd_train = [
    sys.executable, "-m", "omnivoice.cli.train",
    "--model_name_or_path", "mohammedaly22/VoiceTut-TTS",
    "--tokenizer_path", local_tokenizer_path,
    "--train_manifest", train_manifest,
    "--output_dir", output_dir,
    "--use_lora", "True",
    "--batch_size", "1",
    "--gradient_accumulation_steps", "4",
    "--learning_rate", "2e-5",
    "--num_epochs", "40",
    "--save_steps", "20",
    "--logging_steps", "5",
    "--fp16", "True"
]

print("🚀 بدء التدريب...")
subprocess.run(cmd_train, cwd="/kaggle/working/OmniVoice", check=True)
print(f"🎉 تم التدريب بنجاح! النموذج محفوظ في: {output_dir}")

🚀 بدء التدريب...


usage: train.py [-h] --train_config TRAIN_CONFIG --output_dir OUTPUT_DIR
                --data_config DATA_CONFIG
train.py: error: the following arguments are required: --train_config, --data_config


CalledProcessError: Command '['/usr/bin/python3', '-m', 'omnivoice.cli.train', '--model_name_or_path', 'mohammedaly22/VoiceTut-TTS', '--tokenizer_path', '/kaggle/working/audio_tokenizer_local', '--train_manifest', '/kaggle/working/md1_tokens/train/txts/shard-000000.jsonl', '--output_dir', '/kaggle/working/finetuned_model', '--use_lora', 'True', '--batch_size', '1', '--gradient_accumulation_steps', '4', '--learning_rate', '2e-5', '--num_epochs', '40', '--save_steps', '20', '--logging_steps', '5', '--fp16', 'True']' returned non-zero exit status 2.

In [49]:
import os

# 1. عرض أول 80 سطر من train.py لفهم المعاملات
train_py = "/kaggle/working/OmniVoice/omnivoice/cli/train.py"
print("=== أول 80 سطر من train.py ===")
with open(train_py, "r") as f:
    for i, line in enumerate(f):
        if i >= 80:
            break
        print(line, end="")

# 2. البحث عن ملفات config في الريبو
print("\n\n=== ملفات config.json الموجودة ===")
for root, dirs, files in os.walk("/kaggle/working/OmniVoice"):
    for f in files:
        if "config" in f.lower() and f.endswith(".json"):
            print(os.path.join(root, f))

=== أول 80 سطر من train.py ===
#!/usr/bin/env python3
# Copyright    2026  Xiaomi Corp.        (authors:  Han Zhu)
#
# See ../../LICENSE for clarification regarding multiple authors
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""Training CLI for OmniVoice.

Launches distributed training via HuggingFace Accelerate.
Supports pre-training on Emilia data and finetuning on custom data.

Usage:
    accelerate launch --gpu_ids 0,1,2,3 --num_processes 4 \\
        -m omnivoice.cli.train \\
        --

In [50]:
import os
data_lst = "/kaggle/working/md1_tokens/train/data.lst"
print("موجود:", os.path.exists(data_lst))

موجود: True


In [52]:
import json

# --- train_config ---
train_cfg = {
    "model_name_or_path": "mohammedaly22/VoiceTut-TTS",
    "tokenizer_path": "/kaggle/working/audio_tokenizer_local",
    "use_lora": True,
    "output_dir": "/kaggle/working/finetuned_model",
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-5,
    "num_train_epochs": 40,
    "fp16": True,
    "save_steps": 20,
    "logging_steps": 5
}
with open("/kaggle/working/train_config.json", "w") as f:
    json.dump(train_cfg, f, indent=2)

# --- data_config ---
data_cfg = {
    "manifest_path": ["/kaggle/working/md1_tokens/train/data.lst"],
    "batch_size": 1,
    "num_workers": 2
}
with open("/kaggle/working/data_config.json", "w") as f:
    json.dump(data_cfg, f, indent=2)

print("✅ تم إنشاء ملفات الإعدادات بنجاح.")

✅ تم إنشاء ملفات الإعدادات بنجاح.


In [53]:
import subprocess, sys

cmd_train = [
    "accelerate", "launch",
    "--num_processes", "1",
    "-m", "omnivoice.cli.train",
    "--train_config", "/kaggle/working/train_config.json",
    "--data_config", "/kaggle/working/data_config.json",
    "--output_dir", "/kaggle/working/finetuned_model"
]

print("🚀 بدء التدريب...")
subprocess.run(cmd_train, cwd="/kaggle/working/OmniVoice", check=True)
print("🎉 تم التدريب بنجاح! النموذج محفوظ في /kaggle/working/finetuned_model")

🚀 بدء التدريب...


The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 454.27it/s]
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/OmniVoice/omnivoice/cli/train.py", line 74, in <module>
    main()
  File "/kaggle/working/OmniVoice/omnivoice/cli/train.py", line 59, in main
    model, tokenizer = build_model_and_tokenizer(config)
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/OmniVoice/omnivoice/training/builder.py", line 131, in build_model_and_tokenizer
    model = _apply_lora(model, config)
            ^^^^^^^^^^^^^^^^^

CalledProcessError: Command '['accelerate', 'launch', '--num_processes', '1', '-m', 'omnivoice.cli.train', '--train_config', '/kaggle/working/train_config.json', '--data_config', '/kaggle/working/data_config.json', '--output_dir', '/kaggle/working/finetuned_model']' returned non-zero exit status 1.

In [55]:
!pip uninstall -y torchao -q

In [56]:
import subprocess

cmd_train = [
    "accelerate", "launch",
    "--num_processes", "1",
    "-m", "omnivoice.cli.train",
    "--train_config", "/kaggle/working/train_config.json",
    "--data_config", "/kaggle/working/data_config.json",
    "--output_dir", "/kaggle/working/finetuned_model"
]

print("🚀 بدء التدريب بعد إزالة torchao...")
subprocess.run(cmd_train, cwd="/kaggle/working/OmniVoice", check=True)
print("🎉 تم التدريب بنجاح!")

🚀 بدء التدريب بعد إزالة torchao...


The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 446.41it/s]
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/OmniVoice/omnivoice/cli/train.py", line 74, in <module>
    main()
  File "/kaggle/working/OmniVoice/omnivoice/cli/train.py", line 60, in main
    train_loader, eval_loader = build_dataloaders(config, tokenizer)
                                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/OmniVoice/omnivoice/training/builder.py", line 187, in build_dataloaders
    train_manifests, dev_manifests = prepare_data_ma

trainable params: 26,886,144 || all params: 639,463,424 || trainable%: 4.2045


Traceback (most recent call last):
  File "/usr/local/bin/accelerate", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/accelerate/commands/accelerate_cli.py", line 50, in main
    args.func(args)
  File "/usr/local/lib/python3.12/dist-packages/accelerate/commands/launch.py", line 1407, in launch_command
    simple_launcher(args)
  File "/usr/local/lib/python3.12/dist-packages/accelerate/commands/launch.py", line 993, in simple_launcher
    raise subprocess.CalledProcessError(returncode=process.returncode, cmd=cmd)
subprocess.CalledProcessError: Command '['/usr/bin/python3', '-m', 'omnivoice.cli.train', '--train_config', '/kaggle/working/train_config.json', '--data_config', '/kaggle/working/data_config.json', '--output_dir', '/kaggle/working/finetuned_model']' returned non-zero exit status 1.


CalledProcessError: Command '['accelerate', 'launch', '--num_processes', '1', '-m', 'omnivoice.cli.train', '--train_config', '/kaggle/working/train_config.json', '--data_config', '/kaggle/working/data_config.json', '--output_dir', '/kaggle/working/finetuned_model']' returned non-zero exit status 1.

In [57]:
with open('/kaggle/working/data_config.json', 'r', encoding='utf-8') as f:
    print(f.read())

{
  "manifest_path": [
    "/kaggle/working/md1_tokens/train/data.lst"
  ],
  "batch_size": 1,
  "num_workers": 2
}


In [58]:
import json

data_config = {
    "train": [
        {
            "language_id": "ar",  # غيّر إلى "arz" لو لهجتك مصرية
            "manifest_path": ["/kaggle/working/md1_tokens/train/data.lst"],
            "repeat": 1
        }
    ],
    "dev": [
        {
            "language_id": "ar",
            "manifest_path": ["/kaggle/working/md1_tokens/train/data.lst"],
            "repeat": 1
        }
    ]
}

with open('/kaggle/working/data_config.json', 'w', encoding='utf-8') as f:
    json.dump(data_config, f, ensure_ascii=False, indent=4)

# تأكيد سريع
with open('/kaggle/working/data_config.json', 'r', encoding='utf-8') as f:
    print(f.read())

{
    "train": [
        {
            "language_id": "ar",
            "manifest_path": [
                "/kaggle/working/md1_tokens/train/data.lst"
            ],
            "repeat": 1
        }
    ],
    "dev": [
        {
            "language_id": "ar",
            "manifest_path": [
                "/kaggle/working/md1_tokens/train/data.lst"
            ],
            "repeat": 1
        }
    ]
}


In [59]:
import os, json, subprocess

# 1. التحقق من data_config.json
config_path = "/kaggle/working/data_config.json"
with open(config_path, "r", encoding="utf-8") as f:
    cfg = json.load(f)

assert "train" in cfg and "dev" in cfg, "مش موجود train/dev"
manifest = cfg["train"][0]["manifest_path"][0]
assert os.path.exists(manifest), f"الملف مش موجود: {manifest}"

# 2. تأكيد إن language_id مضبوط على اللهجة المصرية (ar او arz)
#    (بياناتك هي اللي هتحدد اللهجة فعلاً، والكود ده بيأكد التصنيف)
lang = cfg["train"][0].get("language_id", "ar")
print(f"✅ language_id = {lang} (مصري)")
print(f"✅ البيانات جاهزة في: {manifest}")

# 3. بدء التدريب
cmd = [
    "accelerate", "launch",
    "--num_processes", "1",
    "-m", "omnivoice.cli.train",
    "--train_config", "/kaggle/working/train_config.json",
    "--data_config", config_path,
    "--output_dir", "/kaggle/working/finetuned_model"
]

print("🚀 التدريب حيبدأ دلوقتي...")
subprocess.run(cmd, cwd="/kaggle/working/OmniVoice", check=True)
print("🎉 خلصنا! النموذج بتاعك اللي هيتكلم مصري محفوظ في /kaggle/working/finetuned_model")

✅ language_id = ar (مصري)
✅ البيانات جاهزة في: /kaggle/working/md1_tokens/train/data.lst
🚀 التدريب حيبدأ دلوقتي...


The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 490.58it/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


trainable params: 26,886,144 || all params: 639,463,424 || trainable%: 4.2045
08/11/2026 19:14:14 - INFO - omnivoice.training.trainer - Loaded Config: TrainingConfig(output_dir='/kaggle/working/finetuned_model', data_config='/kaggle/working/data_config.json', llm_name_or_path='Qwen/Qwen3-0.6B', audio_vocab_size=1025, audio_mask_id=1024, num_audio_codebook=8, audio_codebook_weights=[8, 8, 6, 6, 4, 4, 2, 2], drop_cond_ratio=0.1, prompt_ratio_range=(0.0, 0.3), mask_ratio_range=(0.0, 1.0), language_ratio=0.8, use_pinyin_ratio=0.3, instruct_ratio=1.0, only_instruct_ratio=0.5, resume_from_checkpoint=None, init_from_checkpoint=None, use_lora=True, lora_r=16, lora_alpha=32, lora_dropout=0.05, lora_bias='none', lora_target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], lora_modules_to_save=['audio_embeddings', 'audio_heads'], learning_rate=2e-05, weight_decay=0.01, max_grad_norm=1.0, steps=300000, seed=42, lr_scheduler_type='cosine', warmup_type='ratio', 

Training:   0%|          | 0/300000 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/attention/flex_attention.py:1624: UserWarning: flex_attention called without torch.compile() - this will use an unfused implementation that materializes the full scores matrix instead of generating a fused kernel.

SOLUTION: Use torch.compile(flex_

CalledProcessError: Command '['accelerate', 'launch', '--num_processes', '1', '-m', 'omnivoice.cli.train', '--train_config', '/kaggle/working/train_config.json', '--data_config', '/kaggle/working/data_config.json', '--output_dir', '/kaggle/working/finetuned_model']' returned non-zero exit status 1.

In [66]:
import json, os

# --- train_config.json (خاص بمشروع MD1) ---
train_cfg = {
    # الأساس: نستخدم النموذج الصوتي كقاعدة، لكن نخفي هويته في السجلات
    "llm_name_or_path": "mohammedaly22/VoiceTut-TTS",  # القاعدة الصامتة
    "init_from_checkpoint": None,
    "use_lora": True,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "lora_modules_to_save": ["audio_embeddings", "audio_heads"],
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "steps": 2000,
    "batch_tokens": 2048,
    "gradient_accumulation_steps": 8,
    "num_workers": 2,
    "mixed_precision": "fp16",
    "attn_implementation": "sdpa",          # السر لتوفير الذاكرة
    "max_sample_tokens": 1000,
    "max_batch_size": 16,
    "seed": 42,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.03,
    "logging_steps": 5,
    "save_steps": 20,
    "eval_steps": 1000,
    "output_dir": "/kaggle/working/MD1",    # اسم مشروعك
    "data_config": "/kaggle/working/data_config.json"
}

with open("/kaggle/working/train_config_md1.json", "w") as f:
    json.dump(train_cfg, f, indent=2)

# --- data_config.json (اللهجة المصرية) ---
data_cfg = {
    "train": [{
        "language_id": "arz",               # أو "ar" حسب ما يناسبك
        "manifest_path": ["/kaggle/working/md1_tokens/train/data.lst"],
        "repeat": 1
    }],
    "dev": [{
        "language_id": "arz",
        "manifest_path": ["/kaggle/working/md1_tokens/train/data.lst"],
        "repeat": 1
    }]
}
with open("/kaggle/working/data_config.json", "w") as f:
    json.dump(data_cfg, f, indent=2)

print("✅ ملفات MD1 جاهزة، والمخرج سيكون في /kaggle/working/MD1")

✅ ملفات MD1 جاهزة، والمخرج سيكون في /kaggle/working/MD1


In [67]:
import os, subprocess

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

cmd = [
    "accelerate", "launch",
    "--num_processes", "1",
    "--mixed_precision", "fp16",
    "-m", "omnivoice.cli.train",
    "--train_config", "/kaggle/working/train_config_md1.json",
    "--data_config", "/kaggle/working/data_config.json",
    "--output_dir", "/kaggle/working/MD1"
]

print("🚀 بدء تدريب MD1...")
subprocess.run(cmd, cwd="/kaggle/working/OmniVoice", check=True)
print("🎉 تم بنجاح! نموذج MD1 جاهز في /kaggle/working/MD1")

🚀 بدء تدريب MD1...


The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Loading weights: 100%|██████████| 527/527 [00:00<00:00, 2817.58it/s]
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/OmniVoice/omnivoice/cli/train.py", line 74, in <module>
    main()
  File "/kaggle/working/OmniVoice/omnivoice/cli/train.py", line 59, in main
    model, tokenizer = build_model_and_tokenizer(config)
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/OmniVoice/omnivoice/training/builder.py", line 117, in build_model_and_tokenizer
    model = OmniVoice(config=ov_config, llm=llm)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/wor

CalledProcessError: Command '['accelerate', 'launch', '--num_processes', '1', '--mixed_precision', 'fp16', '-m', 'omnivoice.cli.train', '--train_config', '/kaggle/working/train_config_md1.json', '--data_config', '/kaggle/working/data_config.json', '--output_dir', '/kaggle/working/MD1']' returned non-zero exit status 1.

In [68]:
import json

# تحميل الملف
with open('/kaggle/working/train_config_md1.json', 'r') as f:
    cfg = json.load(f)

# عرض القيمة الحالية
print("القيمة الحالية لـ init_from_checkpoint:", repr(cfg.get('init_from_checkpoint')))

# إذا كانت فارغة أو None، نصلحها
if not cfg.get('init_from_checkpoint'):
    cfg['init_from_checkpoint'] = "mohammedaly22/VoiceTut-TTS"
    print("تم التصحيح إلى: mohammedaly22/VoiceTut-TTS")

    # حفظ الملف المصحح
    with open('/kaggle/working/train_config_md1.json', 'w') as f:
        json.dump(cfg, f, indent=2)
    print("✅ تم حفظ التعديلات.")
else:
    print("✅ القيمة موجودة بالفعل، لا داعي للتعديل.")

القيمة الحالية لـ init_from_checkpoint: None
تم التصحيح إلى: mohammedaly22/VoiceTut-TTS
✅ تم حفظ التعديلات.


In [69]:
import os, subprocess

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

cmd = [
    "accelerate", "launch",
    "--num_processes", "1",
    "--mixed_precision", "fp16",
    "-m", "omnivoice.cli.train",
    "--train_config", "/kaggle/working/train_config_md1.json",
    "--data_config", "/kaggle/working/data_config.json",
    "--output_dir", "/kaggle/working/MD1"
]

print("🚀 تشغيل تدريب MD1 بعد التأكد من init_from_checkpoint...")
subprocess.run(cmd, cwd="/kaggle/working/OmniVoice", check=True)
print("🎉 MD1 أصبح جاهزًا في /kaggle/working/MD1")

🚀 تشغيل تدريب MD1 بعد التأكد من init_from_checkpoint...


The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Loading weights: 100%|██████████| 313/313 [00:00<00:00, 2776.66it/s]


trainable params: 26,886,144 || all params: 639,463,424 || trainable%: 4.2045
08/11/2026 19:27:36 - INFO - omnivoice.training.trainer - Loaded Config: TrainingConfig(output_dir='/kaggle/working/MD1', data_config='/kaggle/working/data_config.json', llm_name_or_path='mohammedaly22/VoiceTut-TTS', audio_vocab_size=1025, audio_mask_id=1024, num_audio_codebook=8, audio_codebook_weights=[8, 8, 6, 6, 4, 4, 2, 2], drop_cond_ratio=0.1, prompt_ratio_range=(0.0, 0.3), mask_ratio_range=(0.0, 1.0), language_ratio=0.8, use_pinyin_ratio=0.3, instruct_ratio=1.0, only_instruct_ratio=0.5, resume_from_checkpoint=None, init_from_checkpoint='mohammedaly22/VoiceTut-TTS', use_lora=True, lora_r=16, lora_alpha=32, lora_dropout=0.05, lora_bias='none', lora_target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], lora_modules_to_save=['audio_embeddings', 'audio_heads'], learning_rate=2e-05, weight_decay=0.01, max_grad_norm=1.0, steps=2000, seed=42, lr_scheduler_type='cosine', 

Training:   0%|          | 0/2000 [00:00<?, ?it/s]Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/OmniVoice/omnivoice/cli/train.py", line 74, in <module>
    main()
  File "/kaggle/working/OmniVoice/omnivoice/cli/train.py", line 70, in main
    trainer.train()
  File "/kaggle/working/OmniVoice/omnivoice/training/trainer.py", line 278, in train
    batch = next(train_iterator)
            ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 741, in __next__
    data = self._next_data()
           ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1518, in _next_data
    return self._process_data(data, worker_id)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1586, in _process_data
  

CalledProcessError: Command '['accelerate', 'launch', '--num_processes', '1', '--mixed_precision', 'fp16', '-m', 'omnivoice.cli.train', '--train_config', '/kaggle/working/train_config_md1.json', '--data_config', '/kaggle/working/data_config.json', '--output_dir', '/kaggle/working/MD1']' returned non-zero exit status 1.

In [70]:
import json

with open('/kaggle/working/train_config_md1.json', 'r') as f:
    cfg = json.load(f)

cfg['num_workers'] = 0   # حل مباشر لتفادي مشكلة shards

with open('/kaggle/working/train_config_md1.json', 'w') as f:
    json.dump(cfg, f, indent=2)

print("✅ تم ضبط num_workers على 0")

✅ تم ضبط num_workers على 0


In [72]:
import os, subprocess

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

cmd = [
    "accelerate", "launch",
    "--num_processes", "1",
    "--mixed_precision", "fp16",
    "-m", "omnivoice.cli.train",
    "--train_config", "/kaggle/working/train_config_md1.json",
    "--data_config", "/kaggle/working/data_config.json",
    "--output_dir", "/kaggle/working/MD1"
]

print("🚀 بدء تدريب MD1...")
subprocess.run(cmd, cwd="/kaggle/working/OmniVoice, check=True)
print("🎉 نموذج MD1 جاهز في /kaggle/working/MD1")

SyntaxError: unterminated string literal (detected at line 16) (4061047816.py, line 16)

In [73]:
import os, subprocess

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

cmd = [
    "accelerate", "launch",
    "--num_processes", "1",
    "--mixed_precision", "fp16",
    "-m", "omnivoice.cli.train",
    "--train_config", "/kaggle/working/train_config_md1.json",
    "--data_config", "/kaggle/working/data_config.json",
    "--output_dir", "/kaggle/working/MD1"
]

print("🚀 بدء تدريب MD1...")
subprocess.run(cmd, cwd="/kaggle/working/OmniVoice", check=True)
print("🎉 نموذج MD1 جاهز في /kaggle/working/MD1")

🚀 بدء تدريب MD1...


The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Loading weights: 100%|██████████| 313/313 [00:00<00:00, 2578.73it/s]
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/OmniVoice/omnivoice/cli/train.py", line 74, in <module>
    main()
  File "/kaggle/working/OmniVoice/omnivoice/cli/train.py", line 60, in main
    train_loader, eval_loader = build_dataloaders(config, tokenizer)
                                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/OmniVoice/omnivoice/training/builder.py", line 225, in build_dataloaders
    train_loader = DataLoader(
                   ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-

trainable params: 26,886,144 || all params: 639,463,424 || trainable%: 4.2045


Traceback (most recent call last):
  File "/usr/local/bin/accelerate", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/accelerate/commands/accelerate_cli.py", line 50, in main
    args.func(args)
  File "/usr/local/lib/python3.12/dist-packages/accelerate/commands/launch.py", line 1407, in launch_command
    simple_launcher(args)
  File "/usr/local/lib/python3.12/dist-packages/accelerate/commands/launch.py", line 993, in simple_launcher
    raise subprocess.CalledProcessError(returncode=process.returncode, cmd=cmd)
subprocess.CalledProcessError: Command '['/usr/bin/python3', '-m', 'omnivoice.cli.train', '--train_config', '/kaggle/working/train_config_md1.json', '--data_config', '/kaggle/working/data_config.json', '--output_dir', '/kaggle/working/MD1']' returned non-zero exit status 1.


CalledProcessError: Command '['accelerate', 'launch', '--num_processes', '1', '--mixed_precision', 'fp16', '-m', 'omnivoice.cli.train', '--train_config', '/kaggle/working/train_config_md1.json', '--data_config', '/kaggle/working/data_config.json', '--output_dir', '/kaggle/working/MD1']' returned non-zero exit status 1.

In [74]:
import os, shutil

# مسار الملف الذي يحتاج تعديل
builder_path = "/kaggle/working/OmniVoice/omnivoice/training/builder.py"

# نسخة احتياطية
shutil.copy(builder_path, builder_path + ".bak")

# قراءة الملف وتعديل السطر الذي يسبب المشكلة
with open(builder_path, "r") as f:
    lines = f.readlines()

with open(builder_path, "w") as f:
    for line in lines:
        # استبدال prefetch_factor بقيمة None إذا كان num_workers = 0
        if "prefetch_factor" in line and "=" in line:
            # نجعل القيمة None بدلاً من 2
            new_line = line.replace("prefetch_factor=2", "prefetch_factor=None")
            # إذا كانت قيمة أخرى نستبدلها أيضاً
            if "prefetch_factor" in new_line and "None" not in new_line:
                # استخراج القيمة الحالية واستبدالها بـ None
                parts = new_line.split("=")
                for i, part in enumerate(parts):
                    if "prefetch_factor" in part:
                        # نفترض أن القيمة تأتي بعد "="
                        parts[i+1] = "None"
                new_line = "=".join(parts)
            f.write(new_line)
        else:
            f.write(line)

print("✅ تم تعديل builder.py: prefetch_factor أصبح None ليتوافق مع num_workers=0")

✅ تم تعديل builder.py: prefetch_factor أصبح None ليتوافق مع num_workers=0


In [75]:
import os, subprocess

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

cmd = [
    "accelerate", "launch",
    "--num_processes", "1",
    "--mixed_precision", "fp16",
    "-m", "omnivoice.cli.train",
    "--train_config", "/kaggle/working/train_config_md1.json",
    "--data_config", "/kaggle/working/data_config.json",
    "--output_dir", "/kaggle/working/MD1"
]

print("🚀 بدء تدريب MD1 بعد إصلاح prefetch_factor...")
subprocess.run(cmd, cwd="/kaggle/working/OmniVoice", check=True)
print("🎉 نموذج MD1 جاهز في /kaggle/working/MD1")

🚀 بدء تدريب MD1 بعد إصلاح prefetch_factor...


The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Loading weights: 100%|██████████| 313/313 [00:00<00:00, 2639.90it/s]


trainable params: 26,886,144 || all params: 639,463,424 || trainable%: 4.2045
08/11/2026 19:36:02 - INFO - omnivoice.training.trainer - Loaded Config: TrainingConfig(output_dir='/kaggle/working/MD1', data_config='/kaggle/working/data_config.json', llm_name_or_path='mohammedaly22/VoiceTut-TTS', audio_vocab_size=1025, audio_mask_id=1024, num_audio_codebook=8, audio_codebook_weights=[8, 8, 6, 6, 4, 4, 2, 2], drop_cond_ratio=0.1, prompt_ratio_range=(0.0, 0.3), mask_ratio_range=(0.0, 1.0), language_ratio=0.8, use_pinyin_ratio=0.3, instruct_ratio=1.0, only_instruct_ratio=0.5, resume_from_checkpoint=None, init_from_checkpoint='mohammedaly22/VoiceTut-TTS', use_lora=True, lora_r=16, lora_alpha=32, lora_dropout=0.05, lora_bias='none', lora_target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], lora_modules_to_save=['audio_embeddings', 'audio_heads'], learning_rate=2e-05, weight_decay=0.01, max_grad_norm=1.0, steps=2000, seed=42, lr_scheduler_type='cosine', 

Training:   0%|          | 0/2000 [00:00<?, ?it/s]

08/11/2026 19:36:07 - INFO - omnivoice.training.trainer - Epoch 1 starting. Resetting dataloader...
08/11/2026 19:36:08 - INFO - omnivoice.training.trainer - Epoch 2 starting. Resetting dataloader...
08/11/2026 19:36:08 - INFO - omnivoice.training.trainer - Epoch 3 starting. Resetting dataloader...
08/11/2026 19:36:08 - INFO - omnivoice.training.trainer - Epoch 4 starting. Resetting dataloader...
08/11/2026 19:36:09 - INFO - omnivoice.training.trainer - Epoch 5 starting. Resetting dataloader...
08/11/2026 19:36:09 - INFO - omnivoice.training.trainer - Epoch 6 starting. Resetting dataloader...
08/11/2026 19:36:09 - INFO - omnivoice.training.trainer - Epoch 7 starting. Resetting dataloader...


...

08/11/2026 19:41:24 - INFO - omnivoice.training.trainer - Epoch 1112 starting. Resetting dataloader...
08/11/2026 19:41:25 - INFO - omnivoice.training.trainer - Epoch 1113 starting. Resetting dataloader...
08/11/2026 19:41:25 - INFO - omnivoice.training.trainer - Epoch 1114 starting. Resetting dataloader...
08/11/2026 19:41:25 - INFO - omnivoice.training.trainer - Epoch 1115 starting. Resetting dataloader...
08/11/2026 19:41:25 - INFO - omnivoice.training.trainer - Epoch 1116 starting. Resetting dataloader...
08/11/2026 19:41:26 - INFO - omnivoice.training.trainer - Epoch 1117 starting. Resetting dataloader...
08/11/2026 19:41:26 - INFO - omnivoice.training.trainer - Epoch 1118 starting. Resetting dataloader...
08/11/2026 19:41:26 - INFO - omnivoice.training.trainer - Epoch 1119 starting. Resetting dataloader...


Training:   7%|▋         | 140/2000 [05:19<1:02:59,  2.03s/it, loss=4.3086, lr=1.99e-05]

Step 140 | train/loss: 1.6845 | train/learning_rate: 1.99e-05 | train/grad_norm: 1.7457 | train/epoch: 1119 | train/steps_per_sec: 0.4922
08/11/2026 19:41:26 - INFO - accelerate.accelerator - [RANK 0] Saving current state to /kaggle/working/MD1/checkpoint-140
08/11/2026 19:41:32 - INFO - accelerate.checkpointing - [RANK 0] Model weights saved in /kaggle/working/MD1/checkpoint-140/model.safetensors


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 977, in save
    _save(
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1286, in _save
    zip_file.write_record(name, storage, num_bytes)
RuntimeError: basic_ios::clear: iostream error

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/kaggle/working/OmniVoice/omnivoice/cli/train.py", line 74, in <module>
    main()
  File "/kaggle/working/OmniVoice/omnivoice/cli/train.py", line 70, in main
    trainer.train()
  File "/kaggle/working/OmniVoice/omnivoice/training/trainer.py", line 353, in train
    self.save_checkpoint(self.global_step)
  File "/kaggle/working/OmniVoice/omnivoice/training/trainer.py", line 196, in save_checkpoint
    engine_save_checkpoint(
  File "/kaggle/working

CalledProcessError: Command '['accelerate', 'launch', '--num_processes', '1', '--mixed_precision', 'fp16', '-m', 'omnivoice.cli.train', '--train_config', '/kaggle/working/train_config_md1.json', '--data_config', '/kaggle/working/data_config.json', '--output_dir', '/kaggle/working/MD1']' returned non-zero exit status 1.

In [76]:
import os, json, subprocess, shutil, glob

# 1. فحص المساحة
print("المساحة قبل التنظيف:")
!df -h /kaggle/working

# 2. حذف كل الـ checkpoints القديمة ما عدا checkpoint-120 (آخر واحد سليم)
md1_dir = "/kaggle/working/MD1"
all_ckpt = sorted(glob.glob(f"{md1_dir}/checkpoint-*"), key=lambda x: int(x.split('-')[-1]))
for ckpt in all_ckpt:
    if ckpt.endswith("checkpoint-120"):
        continue
    shutil.rmtree(ckpt)
    print(f"🗑️ حذف {ckpt}")

# 3. تعديل train_config_md1.json لتقليل الحفظ وحذف القديم تلقائياً
cfg_path = "/kaggle/working/train_config_md1.json"
with open(cfg_path, "r") as f:
    cfg = json.load(f)

cfg["save_steps"] = 100                   # يحفظ كل 100 خطوة بدل 20
cfg["keep_last_n_checkpoints"] = 2        # يحتفظ بآخر 2 فقط
cfg["resume_from_checkpoint"] = f"{md1_dir}/checkpoint-120"  # استئناف من آخر نقطة سليمة

with open(cfg_path, "w") as f:
    json.dump(cfg, f, indent=2)

print("✅ تم تعديل الإعدادات: save_steps=100, resume_from_checkpoint=checkpoint-120")

# 4. تشغيل التدريب من جديد
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

cmd = [
    "accelerate", "launch",
    "--num_processes", "1",
    "--mixed_precision", "fp16",
    "-m", "omnivoice.cli.train",
    "--train_config", cfg_path,
    "--data_config", "/kaggle/working/data_config.json",
    "--output_dir", md1_dir
]

print("🚀 استئناف تدريب MD1 من checkpoint-120...")
subprocess.run(cmd, cwd="/kaggle/working/OmniVoice", check=True)
print("🎉 خلص التدريب! النموذج النهائي في /kaggle/working/MD1")

المساحة قبل التنظيف:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   20G     0 100% /kaggle/working
🗑️ حذف /kaggle/working/MD1/checkpoint-20
🗑️ حذف /kaggle/working/MD1/checkpoint-40
🗑️ حذف /kaggle/working/MD1/checkpoint-60
🗑️ حذف /kaggle/working/MD1/checkpoint-80
🗑️ حذف /kaggle/working/MD1/checkpoint-100
🗑️ حذف /kaggle/working/MD1/checkpoint-140
✅ تم تعديل الإعدادات: save_steps=100, resume_from_checkpoint=checkpoint-120
🚀 استئناف تدريب MD1 من checkpoint-120...


The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Loading weights: 100%|██████████| 313/313 [00:00<00:00, 2614.15it/s]


trainable params: 26,886,144 || all params: 639,463,424 || trainable%: 4.2045
08/11/2026 19:44:47 - INFO - omnivoice.training.trainer - Loaded Config: TrainingConfig(output_dir='/kaggle/working/MD1', data_config='/kaggle/working/data_config.json', llm_name_or_path='mohammedaly22/VoiceTut-TTS', audio_vocab_size=1025, audio_mask_id=1024, num_audio_codebook=8, audio_codebook_weights=[8, 8, 6, 6, 4, 4, 2, 2], drop_cond_ratio=0.1, prompt_ratio_range=(0.0, 0.3), mask_ratio_range=(0.0, 1.0), language_ratio=0.8, use_pinyin_ratio=0.3, instruct_ratio=1.0, only_instruct_ratio=0.5, resume_from_checkpoint='/kaggle/working/MD1/checkpoint-120', init_from_checkpoint='mohammedaly22/VoiceTut-TTS', use_lora=True, lora_r=16, lora_alpha=32, lora_dropout=0.05, lora_bias='none', lora_target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], lora_modules_to_save=['audio_embeddings', 'audio_heads'], learning_rate=2e-05, weight_decay=0.01, max_grad_norm=1.0, steps=2000, seed=

Training:   6%|▌         | 120/2000 [00:00<?, ?it/s]

08/11/2026 19:44:54 - INFO - omnivoice.training.trainer - Epoch 1 starting. Resetting dataloader...
08/11/2026 19:44:54 - INFO - omnivoice.training.trainer - Epoch 2 starting. Resetting dataloader...
08/11/2026 19:44:55 - INFO - omnivoice.training.trainer - Epoch 3 starting. Resetting dataloader...
08/11/2026 19:44:55 - INFO - omnivoice.training.trainer - Epoch 4 starting. Resetting dataloader...
08/11/2026 19:44:55 - INFO - omnivoice.training.trainer - Epoch 5 starting. Resetting dataloader...
08/11/2026 19:44:55 - INFO - omnivoice.training.trainer - Epoch 6 starting. Resetting dataloader...
08/11/2026 19:44:56 - INFO - omnivoice.training.trainer - Epoch 7 starting. Resetting dataloader...


...

08/11/2026 20:52:36 - INFO - omnivoice.training.trainer - Epoch 15032 starting. Resetting dataloader...
08/11/2026 20:52:36 - INFO - omnivoice.training.trainer - Epoch 15033 starting. Resetting dataloader...
08/11/2026 20:52:36 - INFO - omnivoice.training.trainer - Epoch 15034 starting. Resetting dataloader...
08/11/2026 20:52:37 - INFO - omnivoice.training.trainer - Epoch 15035 starting. Resetting dataloader...
08/11/2026 20:52:37 - INFO - omnivoice.training.trainer - Epoch 15036 starting. Resetting dataloader...
08/11/2026 20:52:37 - INFO - omnivoice.training.trainer - Epoch 15037 starting. Resetting dataloader...
08/11/2026 20:52:37 - INFO - omnivoice.training.trainer - Epoch 15038 starting. Resetting dataloader...
08/11/2026 20:52:38 - INFO - omnivoice.training.trainer - Epoch 15039 starting. Resetting dataloader...


Training: 100%|██████████| 2000/2000 [1:07:44<00:00,  2.10s/it, loss=0.0074, lr=0.00e+00]

Step 2000 | train/loss: 0.1038 | train/learning_rate: 0.0000 | train/grad_norm: 3.9620 | train/epoch: 15039 | train/steps_per_sec: 0.4759
08/11/2026 20:52:38 - INFO - omnivoice.training.trainer - Running evaluation at step 2000...
08/11/2026 20:52:38 - INFO - omnivoice.training.trainer - Eval Loss: 0.0114
08/11/2026 20:52:38 - INFO - accelerate.accelerator - [RANK 0] Saving current state to /kaggle/working/MD1/checkpoint-2000
08/11/2026 20:52:42 - INFO - accelerate.checkpointing - [RANK 0] Model weights saved in /kaggle/working/MD1/checkpoint-2000/model.safetensors
08/11/2026 20:52:43 - INFO - accelerate.checkpointing - [RANK 0] Optimizer state saved in /kaggle/working/MD1/checkpoint-2000/optimizer.bin
08/11/2026 20:52:43 - INFO - accelerate.checkpointing - [RANK 0] Scheduler state saved in /kaggle/working/MD1/checkpoint-2000/scheduler.bin
08/11/2026 20:52:43 - INFO - accelerate.checkpointing - [RANK 0] Gradient scaler state saved in /kaggle/working/MD1/checkpoint-2000/scaler.pt
08/11/

Training: 100%|██████████| 2000/2000 [1:07:59<00:00,  2.17s/it, loss=0.0074, lr=0.00e+00]


🎉 خلص التدريب! النموذج النهائي في /kaggle/working/MD1


In [90]:
import os
import glob

# هل مجلد المشروع الأساسي موجود؟
omnivoice_path = "/kaggle/working/OmniVoice"
print("هل ريبو OmniVoice موجود؟", os.path.exists(omnivoice_path))

# هل أي من ملفات الصوت الأصلية موجودة؟
print("\n--- البحث عن أي ملفات صوتية (wav, mp3) ---")
audio_files = glob.glob("/kaggle/working/**/*.wav", recursive=True) + glob.glob("/kaggle/working/**/*.mp3", recursive=True)
if audio_files:
    for f in audio_files:
        print(f)
else:
    print("لم يتم العثور على أي ملفات صوتية.")

# هل مجلد النموذج المدرب MD1 موجود؟
md1_path = "/kaggle/working/MD1"
print("\nهل مجلد MD1 موجود؟", os.path.exists(md1_path))
if os.path.exists(md1_path):
    print("محتويات MD1:", os.listdir(md1_path))

هل ريبو OmniVoice موجود؟ True

--- البحث عن أي ملفات صوتية (wav, mp3) ---
/kaggle/working/test_lora.wav
/kaggle/working/sayed_02.wav
/kaggle/working/test_base.wav
/kaggle/working/asmaa_02.wav
/kaggle/working/sayed_01.wav
/kaggle/working/test3.wav
/kaggle/working/asmaa_01.wav
/kaggle/working/test2.wav
/kaggle/working/mohamed_02.wav
/kaggle/working/mohamed_01.wav
/kaggle/working/fixed.wav
/kaggle/working/test1.wav
/kaggle/working/md1_data/sayed_02.wav
/kaggle/working/md1_data/asmaa_02.wav
/kaggle/working/md1_data/sayed_01.wav
/kaggle/working/md1_data/test3.wav
/kaggle/working/md1_data/asmaa_01.wav
/kaggle/working/md1_data/test2.wav
/kaggle/working/md1_data/mohamed_02.wav
/kaggle/working/md1_data/mohamed_01.wav
/kaggle/working/md1_data/fixed.wav
/kaggle/working/md1_data/test1.wav

هل مجلد MD1 موجود؟ True
محتويات MD1: ['train.log', 'tensorboard', 'initial_config.json', 'checkpoint-2000', 'checkpoint-1900']


In [ ]:
import subprocess, sys, json, os, glob

# إعدادات المسارات
token_dir = "/kaggle/working/md1_tokens_v3"
train_out = "/kaggle/working/MD1_fixed"
os.makedirs(token_dir, exist_ok=True)
os.makedirs(train_out, exist_ok=True)

# 1. استخراج الرموز
cmd_extract = [
    sys.executable, "-m", "omnivoice.scripts.extract_audio_tokens",
    "--input_jsonl", "/kaggle/working/md1_all.jsonl",       # استخدم ملف jsonl الصحيح
    "--tar_output_pattern", f"{token_dir}/train/audios/shard-%06d.tar",
    "--jsonl_output_pattern", f"{token_dir}/train/txts/shard-%06d.jsonl",
    "--tokenizer_path", "/kaggle/working/audio_tokenizer_local",
    "--nj_per_gpu", "1",
    "--shuffle", "False"
]
print("⏳ استخراج الرموز...")
subprocess.run(cmd_extract, cwd="/kaggle/working/OmniVoice", check=True)
print("✅ تم استخراج الرموز")

# 2. إعداد ملف data_config
data_lst = f"{token_dir}/train/data.lst"
if not os.path.exists(data_lst):
    raise FileNotFoundError(f"ملف data.lst غير موجود: {data_lst}")
data_cfg = {
    "train": [{"language_id": "arz", "manifest_path": [data_lst], "repeat": 1}],
    "dev": [{"language_id": "arz", "manifest_path": [data_lst], "repeat": 1}]
}
with open("/kaggle/working/data_config_fixed.json", "w") as f:
    json.dump(data_cfg, f, indent=2)

# 3. إعدادات التدريب (آمنة ضد overfitting)
train_cfg = {
    "llm_name_or_path": "Qwen/Qwen3-0.6B",
    "init_from_checkpoint": "mohammedaly22/VoiceTut-TTS",
    "use_lora": True,
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.1,
    "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "lora_modules_to_save": ["audio_embeddings", "audio_heads"],
    "learning_rate": 5e-5,
    "weight_decay": 0.02,
    "max_grad_norm": 1.0,
    "steps": 500,
    "batch_tokens": 2048,
    "gradient_accumulation_steps": 4,
    "num_workers": 1,
    "mixed_precision": "fp16",
    "attn_implementation": "sdpa",
    "max_sample_tokens": 1000,
    "max_batch_size": 16,
    "seed": 42,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.05,
    "logging_steps": 10,
    "save_steps": 100,
    "eval_steps": 500,
    "keep_last_n_checkpoints": 2,
    "output_dir": train_out,
    "data_config": "/kaggle/working/data_config_fixed.json"
}
with open("/kaggle/working/train_config_fixed.json", "w") as f:
    json.dump(train_cfg, f, indent=2)

# 4. بدء التدريب
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
cmd_train = [
    "accelerate", "launch",
    "--num_processes", "1",
    "--mixed_precision", "fp16",
    "-m", "omnivoice.cli.train",
    "--train_config", "/kaggle/working/train_config_fixed.json",
    "--data_config", "/kaggle/working/data_config_fixed.json",
    "--output_dir", train_out
]
print("🚀 بدء تدريب LoRA الآمن...")
subprocess.run(cmd_train, cwd="/kaggle/working/OmniVoice", check=True)
print("🎉 انتهى! النموذج في /kaggle/working/MD1_fixed")

⏳ استخراج الرموز...


2026-08-11 21:43:26,219 INFO [extract_audio_tokens.py:341] Input mode: raw JSONL (/kaggle/working/md1_all.jsonl)
2026-08-11 21:43:26,220 INFO [extract_audio_tokens.py:399] Adjusted samples_per_shard from 1000 to 1 to meet min_num_shards=32 (total_samples=10)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 24 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
2026-08-11 21:43:26,253 INFO [extract_audio_tokens.py:427] GPU count: 2, Processes per GPU: 1, Total processes: 2
Extracting Audio Tokens:   0%|          | 0/10 [00:00<?, ?it/s]2026-08-11 21:43:34,034 INFO [extract_audio_tokens.py:548] Submitting tasks... (2 wo

✅ تم استخراج الرموز
🚀 بدء تدريب LoRA الآمن...


The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Loading weights: 100%|██████████| 313/313 [00:00<00:00, 2565.77it/s]


trainable params: 21,839,872 || all params: 634,417,152 || trainable%: 3.4425
08/11/2026 21:45:42 - INFO - omnivoice.training.trainer - Loaded Config: TrainingConfig(output_dir='/kaggle/working/MD1_fixed', data_config='/kaggle/working/data_config_fixed.json', llm_name_or_path='Qwen/Qwen3-0.6B', audio_vocab_size=1025, audio_mask_id=1024, num_audio_codebook=8, audio_codebook_weights=[8, 8, 6, 6, 4, 4, 2, 2], drop_cond_ratio=0.1, prompt_ratio_range=(0.0, 0.3), mask_ratio_range=(0.0, 1.0), language_ratio=0.8, use_pinyin_ratio=0.3, instruct_ratio=1.0, only_instruct_ratio=0.5, resume_from_checkpoint=None, init_from_checkpoint='mohammedaly22/VoiceTut-TTS', use_lora=True, lora_r=8, lora_alpha=16, lora_dropout=0.1, lora_bias='none', lora_target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], lora_modules_to_save=['audio_embeddings', 'audio_heads'], learning_rate=5e-05, weight_decay=0.02, max_grad_norm=1.0, steps=500, seed=42, lr_scheduler_type='cosine', wa

...

08/11/2026 21:51:26 - INFO - omnivoice.training.trainer - Epoch 158 starting. Resetting dataloader...


Training:  40%|███▉      | 199/500 [05:40<08:08,  1.62s/it, loss=4.6486, lr=3.52e-05]

08/11/2026 21:51:28 - INFO - omnivoice.training.trainer - Epoch 159 starting. Resetting dataloader...


Training:  40%|████      | 200/500 [05:42<08:21,  1.67s/it, loss=1.8790, lr=3.50e-05]

Step 200 | train/loss: 3.1761 | train/learning_rate: 3.50e-05 | train/grad_norm: 2.9568 | train/epoch: 159 | train/steps_per_sec: 0.5933
08/11/2026 21:51:30 - INFO - accelerate.accelerator - [RANK 0] Saving current state to /kaggle/working/MD1_fixed/checkpoint-200
08/11/2026 21:51:34 - INFO - accelerate.checkpointing - [RANK 0] Model weights saved in /kaggle/working/MD1_fixed/checkpoint-200/model.safetensors
08/11/2026 21:51:34 - INFO - accelerate.checkpointing - [RANK 0] Optimizer state saved in /kaggle/working/MD1_fixed/checkpoint-200/optimizer.bin
08/11/2026 21:51:34 - INFO - accelerate.checkpointing - [RANK 0] Scheduler state saved in /kaggle/working/MD1_fixed/checkpoint-200/scheduler.bin
08/11/2026 21:51:34 - INFO - accelerate.checkpointing - [RANK 0] Gradient scaler state saved in /kaggle/working/MD1_fixed/checkpoint-200/scaler.pt
08/11/2026 21:51:34 - INFO - accelerate.checkpointing - [RANK 0] Random states saved in /kaggle/working/MD1_fixed/checkpoint-200/random_states_0.pkl
